In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# Define the root directory
root_dir = r'C:\Users\duaas\Downloads\QC'

# Define the subdirectories
intact_top_dir = os.path.join(root_dir, 'intact', 'top')
intact_side_dir = os.path.join(root_dir, 'intact', 'side')
damaged_top_dir = os.path.join(root_dir, 'damaged', 'top')
damaged_side_dir = os.path.join(root_dir, 'damaged', 'side')

# Function to load images from a directory
def load_images(directory):
    images = []
    labels = []
    for filename in os.listdir(directory):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img_path = os.path.join(directory, filename)
            img = cv2.imread(img_path)
            if img is not None:
                images.append(img)
                labels.append(1 if 'damaged' in directory else 0)  # Assign label 1 for damaged, 0 for intact
    return images, labels

# Load images from each category
intact_top_images, intact_top_labels = load_images(intact_top_dir)
intact_side_images, intact_side_labels = load_images(intact_side_dir)
damaged_top_images, damaged_top_labels = load_images(damaged_top_dir)
damaged_side_images, damaged_side_labels = load_images(damaged_side_dir)

# Concatenate images and labels
images = np.concatenate([intact_top_images, intact_side_images, damaged_top_images, damaged_side_images])
labels = np.concatenate([intact_top_labels, intact_side_labels, damaged_top_labels, damaged_side_labels])

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

# Print shapes of the datasets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


C:\Users\duaas\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:177: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Shape of X_train: (320, 540, 960, 3)
Shape of X_test: (80, 540, 960, 3)
Shape of y_train: (320,)
Shape of y_test: (80,)


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Preprocess images
def preprocess_images(images):
    processed_images = []
    for img in images:
        resized_img = cv2.resize(img, (100, 100))  # Resize images to 100x100 pixels
        normalized_img = resized_img / 255.0  # Normalize pixel values to [0, 1]
        processed_images.append(normalized_img)
    return np.array(processed_images)

X_train_processed = preprocess_images(X_train)
X_test_processed = preprocess_images(X_test)

# Define CNN model
model = Sequential()
model.add(Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=(100, 100, 3)))
model.add(MaxPooling2D(pool_size=2))
model.add(Conv2D(filters=64, kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2))
model.add(Conv2D(filters=128, kernel_size=3, activation='relu'))
model.add(MaxPooling2D(pool_size=2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 98, 98, 32)        896       
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 49, 49, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_4 (Conv2D)           (None, 47, 47, 64)        18496     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 23, 23, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 21, 21, 128)       73856     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 10, 10, 128)      

In [4]:
# Train the model
history = model.fit(X_train_processed, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_processed, y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


Epoch 1/10


8/8 [==============================] - 5s 246ms/step - loss: 0.7716 - accuracy: 0.4648 - val_loss: 0.6940 - val_accuracy: 0.4844
Epoch 2/10
8/8 [==============================] - 1s 154ms/step - loss: 0.6930 - accuracy: 0.5195 - val_loss: 0.6938 - val_accuracy: 0.4844
Epoch 3/10
8/8 [==============================] - 1s 163ms/step - loss: 0.6933 - accuracy: 0.4766 - val_loss: 0.6933 - val_accuracy: 0.4844
Epoch 4/10
8/8 [==============================] - 1s 157ms/step - loss: 0.6936 - accuracy: 0.4531 - val_loss: 0.6933 - val_accuracy: 0.4844
Epoch 5/10
8/8 [==============================] - 1s 158ms/step - loss: 0.6936 - accuracy: 0.5195 - val_loss: 0.6965 - val_accuracy: 0.4844
Epoch 6/10
8/8 [==============================] - 1s 156ms/step - loss: 0.6945 - accuracy: 0.4727 - val_loss: 0.6929 - val_accuracy: 0.5625
Epoch 7/10
8/8 [==============================] - 1s 156ms/step - loss: 0.6917 - accuracy: 0.5312 - val_loss: 0.6944 - val_accuracy: 0.4844
Epoch 8/10
8/8 [==

In [6]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define the root directory
root_dir = 'C:/Users/duaas/Downloads/QC/'

# Define image dimensions and batch size
img_width, img_height = 100, 100
batch_size = 32

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Load and augment training data
train_generator = train_datagen.flow_from_directory(
    root_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

# Load validation data without augmentation
validation_generator = train_datagen.flow_from_directory(
    root_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# Define CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_width, img_height, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(validation_generator)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)


Found 320 images belonging to 2 classes.
Found 80 images belonging to 2 classes.
Epoch 1/10
10/10 [==============================] - 8s 713ms/step - loss: 0.7123 - accuracy: 0.5156 - val_loss: 0.6934 - val_accuracy: 0.5156
Epoch 2/10
10/10 [==============================] - 12s 1s/step - loss: 0.7006 - accuracy: 0.4719 - val_loss: 0.6931 - val_accuracy: 0.5469
Epoch 3/10
10/10 [==============================] - 7s 629ms/step - loss: 0.6928 - accuracy: 0.5063 - val_loss: 0.6927 - val_accuracy: 0.5312
Epoch 4/10
10/10 [==============================] - 6s 599ms/step - loss: 0.6937 - accuracy: 0.5250 - val_loss: 0.6934 - val_accuracy: 0.4844
Epoch 5/10
10/10 [==============================] - 6s 608ms/step - loss: 0.6936 - accuracy: 0.5125 - val_loss: 0.6930 - val_accuracy: 0.5312
Epoch 6/10
10/10 [==============================] - 12s 1s/step - loss: 0.6949 - accuracy: 0.4812 - val_loss: 0.6936 - val_accuracy: 0.4844
Epoch 7/10
10/10 [==============================] - 16s 2s/step - loss:

In [7]:
# Assuming 'model' is your trained Keras model
model.save('C:/Users/duaas/QC_miniproject')


INFO:tensorflow:Assets written to: C:/Users/duaas/QC_miniproject\assets


INFO:tensorflow:Assets written to: C:/Users/duaas/QC_miniproject\assets


In [8]:
model = load_model('C:/Users/duaas/QC_miniproject')


In [20]:
import tkinter as tk
from tkinter import filedialog
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('C:/Users/duaas/QC_miniproject')

# Function to preprocess the image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    # Resize the image to match the input shape of the model
    img = cv2.resize(img, (100, 100))
    img = img.astype('float32') / 255.0  # Normalize the pixel values
    return img

# Function to make predictions
def predict_image(image_path):
    preprocessed_img = preprocess_image(image_path)
    # Add a batch dimension and make prediction
    prediction = model.predict(np.expand_dims(preprocessed_img, axis=0))[0]
    if prediction >= 0.5:
        return "Damaged"
    else:
        return "Intact"

# Function to open file dialog and get the image path
def select_image():
    file_path = filedialog.askopenfilename()
    if file_path:
        result_label.config(text="")
        result = predict_image(file_path)
        result_label.config(text="Result: " + result)

# Create a Tkinter window
window = tk.Tk()
window.title("Image Classifier")

# Create a button to select an image
select_button = tk.Button(window, text="Select Image", command=select_image)
select_button.pack(pady=20)

# Label to display the result
result_label = tk.Label(window, text="")
result_label.pack(pady=10)

# Run the Tkinter event loop
window.mainloop()


1/1 [==============================] - 0s 31ms/step


In [2]:
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import numpy as np
import cv2

# Function to preprocess the image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (100, 100))  # Resize to match model input size
    img = img / 255.0  # Normalize pixel values
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img

# Function to classify the image
def classify_image(image_path):
    # Preprocess the image
    img = preprocess_image(image_path)
    # Perform classification using the trained model (Assuming 'model' is already defined)
    # prediction = model.predict(img)
    # Assuming 0 represents 'intact' and 1 represents 'damaged'
    # predicted_class = "intact" if prediction[0][0] == 0 else "damaged"
    # Return the predicted class
    predicted_class = "damaged"  # Temporary placeholder, replace with actual classification
    return predicted_class

# Function to open file dialog, display the selected image, and classify it
def open_image():
    file_path = filedialog.askopenfilename()
    if file_path:
        # Open and display the image
        img = Image.open(file_path)
        img.thumbnail((400, 400))  # Adjust image preview size
        img = ImageTk.PhotoImage(img)
        image_label.config(image=img)
        image_label.image = img
        # Classify the image
        result = classify_image(file_path)
        # Update the result label
        result_label.config(text=f"Classification Result: {result.capitalize()}")

# Create the main window
root = tk.Tk()
root.title("Medicine Package Classification")

# Get the screen width and height
screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()

# Set the window size to the screen size
root.geometry(f"{screen_width}x{screen_height}")

# Create a label to display the selected image
image_label = tk.Label(root)
image_label.pack(padx=10, pady=10)

# Create a button to open the file dialog
open_button = tk.Button(root, text="Open Image", command=open_image)
open_button.pack(pady=10)

# Create a label to display the classification result
result_label = tk.Label(root, text="")
result_label.pack(pady=10)

# Run the application
root.mainloop()
